# Etapa 4 - Treinamento de Modelos

## Objetivo
Nesta etapa vamos treinar modelos de classificação e comparar o desempenho para prever churn.

## O que vamos fazer?
1. carregar os dados prontos do preprocessamento
2. separar features e alvo
3. treinar modelos iniciais
4. comparar métricas
5. escolher o melhor modelo
6. salvar o modelo final

## Modelos que vamos testar
- Regressão Logística
- Random Forest

## Importante
Em problemas de churn, não basta olhar só acurácia. Também precisamos olhar recall, precisão, F1 e ROC-AUC.

In [44]:
# %%
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from xgboost import XGBClassifier

import joblib

print("✓ Bibliotecas importadas com sucesso!")

✓ Bibliotecas importadas com sucesso!


In [45]:
# %%
# Adicione estas linhas na sua célula de importações:

# IMPORTANTE: Requer instalação do imbalanced-learn
# pip install imbalanced-learn
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

print("✓ Novas bibliotecas importadas com sucesso!")

✓ Novas bibliotecas importadas com sucesso!


In [46]:
# %%
# ============================================================
# CONFIGURAÇÃO DOS CAMINHOS
# ============================================================

try:
    notebook_dir = Path.cwd()

    if notebook_dir.name == "notebooks":
        project_root = notebook_dir.parent
    else:
        project_root = notebook_dir

except Exception:
    project_root = Path.cwd()


processed_dir = project_root / "data" / "processed"

artifacts_dir = project_root / "artifacts"
models_dir = artifacts_dir / "models"
tables_dir = artifacts_dir / "tables"


# Criar diretórios caso não existam
models_dir.mkdir(
    parents=True,
    exist_ok=True
)

tables_dir.mkdir(
    parents=True,
    exist_ok=True
)


# Arquivos de entrada
train_path = (
    processed_dir /
    "03_X_train_preprocessed.csv"
)

test_path = (
    processed_dir /
    "04_X_test_preprocessed.csv"
)


print("=" * 60)
print("CONFIGURAÇÃO DA ETAPA 4")
print("=" * 60)

print(f"\nProjeto:")
print(project_root)

print(f"\nDiretório de dados processados:")
print(processed_dir)

print(f"\nArquivo de treino:")
print(train_path)

print(f"\nArquivo de teste:")
print(test_path)

print(f"\nDiretório de modelos:")
print(models_dir)

print(f"\nTreino encontrado: {train_path.exists()}")
print(f"Teste encontrado: {test_path.exists()}")

CONFIGURAÇÃO DA ETAPA 4

Projeto:
c:\Temp\telco-churn-ml

Diretório de dados processados:
c:\Temp\telco-churn-ml\data\processed

Arquivo de treino:
c:\Temp\telco-churn-ml\data\processed\03_X_train_preprocessed.csv

Arquivo de teste:
c:\Temp\telco-churn-ml\data\processed\04_X_test_preprocessed.csv

Diretório de modelos:
c:\Temp\telco-churn-ml\artifacts\models

Treino encontrado: True
Teste encontrado: True


In [47]:
# %%
# ============================================================
# VALIDAÇÃO DOS ARQUIVOS DE ENTRADA
# ============================================================

if not train_path.exists():
    raise FileNotFoundError(
        f"Arquivo de treino não encontrado:\n{train_path}"
    )

if not test_path.exists():
    raise FileNotFoundError(
        f"Arquivo de teste não encontrado:\n{test_path}"
    )

print("✓ Arquivos de treino e teste encontrados.")

✓ Arquivos de treino e teste encontrados.


In [48]:
# %%
# ============================================================
# CARREGAMENTO DOS DADOS
# ============================================================

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("=" * 60)
print("DADOS DE TREINO E TESTE")
print("=" * 60)

print("\nDimensões do treino:")
print(train_df.shape)

print("\nDimensões do teste:")
print(test_df.shape)

print("\nPrimeiras linhas do treino:")
display(train_df.head())

DADOS DE TREINO E TESTE

Dimensões do treino:
(5616, 46)

Dimensões do teste:
(1405, 46)

Primeiras linhas do treino:


,Churn,0,1,2,3,4,5,6,7,8,...,35,36,37,38,39,40,41,42,43,44
0,1,-0.440315,-1.241331,0.193165,-0.946349,0.0,1.0,1.0,0.0,1.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
1,0,-0.440315,-0.711078,0.647355,-0.435345,1.0,0.0,1.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
2,0,-0.440315,1.409933,0.820380,1.794547,0.0,1.0,0.0,1.0,0.0,...,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
3,0,-0.440315,-1.118965,0.023467,-0.858745,0.0,1.0,1.0,0.0,1.0,...,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
4,0,-0.440315,-0.262403,-1.483847,-0.783388,0.0,1.0,1.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


In [49]:
# %%
# ============================================================
# VERIFICAÇÃO DA VARIÁVEL-ALVO
# ============================================================

print("=" * 60)
print("VERIFICAÇÃO DA VARIÁVEL-ALVO")
print("=" * 60)

print("\nColunas do treino:")
print(train_df.columns.tolist())

print("\nColuna Churn existe no treino?")
print("Churn" in train_df.columns)

print("\nColuna Churn existe no teste?")
print("Churn" in test_df.columns)


if "Churn" not in train_df.columns:
    raise ValueError(
        "A coluna 'Churn' não foi encontrada no conjunto de treino."
    )

if "Churn" not in test_df.columns:
    raise ValueError(
        "A coluna 'Churn' não foi encontrada no conjunto de teste."
    )

print("\n✓ Variável-alvo encontrada nos dois conjuntos.")

VERIFICAÇÃO DA VARIÁVEL-ALVO

Colunas do treino:
['Churn', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44']

Coluna Churn existe no treino?
True

Coluna Churn existe no teste?
True

✓ Variável-alvo encontrada nos dois conjuntos.


In [50]:
# %%
# ============================================================
# SEPARAÇÃO ENTRE FEATURES E TARGET
# ============================================================

X_train = train_df.drop(
    columns=["Churn"]
)

y_train = train_df["Churn"]

X_test = test_df.drop(
    columns=["Churn"]
)

y_test = test_df["Churn"]


print("=" * 60)
print("SEPARAÇÃO DE FEATURES E TARGET")
print("=" * 60)

print(f"\nX_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")

print(f"\nX_test: {X_test.shape}")
print(f"y_test: {y_test.shape}")

SEPARAÇÃO DE FEATURES E TARGET

X_train: (5616, 45)
y_train: (5616,)

X_test: (1405, 45)
y_test: (1405,)


In [51]:
# %%
# ============================================================
# DISTRIBUIÇÃO DO TARGET
# ============================================================

print("=" * 60)
print("DISTRIBUIÇÃO DO TARGET")
print("=" * 60)

print("\nTreino:")
print(y_train.value_counts())

print("\nPercentual no treino:")
print(
    y_train.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nTeste:")
print(y_test.value_counts())

print("\nPercentual no teste:")
print(
    y_test.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

DISTRIBUIÇÃO DO TARGET

Treino:
Churn
0    4131
1    1485
Name: count, dtype: int64

Percentual no treino:
Churn
0    73.56
1    26.44
Name: proportion, dtype: float64

Teste:
Churn
0    1033
1     372
Name: count, dtype: int64

Percentual no teste:
Churn
0    73.52
1    26.48
Name: proportion, dtype: float64


In [52]:
# %%
# ============================================================
# VALIDAÇÃO DAS FEATURES
# ============================================================

print("=" * 60)
print("VALIDAÇÃO DAS FEATURES")
print("=" * 60)

print("\nTipos de dados:")
print(X_train.dtypes.value_counts())

print("\nValores ausentes no treino:")
print(X_train.isnull().sum().sum())

print("\nValores ausentes no teste:")
print(X_test.isnull().sum().sum())

print("\nFeatures utilizadas:")
print(f"Total: {X_train.shape[1]}")

for col in X_train.columns:
    print(f" - {col}")

VALIDAÇÃO DAS FEATURES

Tipos de dados:
float64    45
Name: count, dtype: int64

Valores ausentes no treino:
0

Valores ausentes no teste:
0

Features utilizadas:
Total: 45
 - 0
 - 1
 - 2
 - 3
 - 4
 - 5
 - 6
 - 7
 - 8
 - 9
 - 10
 - 11
 - 12
 - 13
 - 14
 - 15
 - 16
 - 17
 - 18
 - 19
 - 20
 - 21
 - 22
 - 23
 - 24
 - 25
 - 26
 - 27
 - 28
 - 29
 - 30
 - 31
 - 32
 - 33
 - 34
 - 35
 - 36
 - 37
 - 38
 - 39
 - 40
 - 41
 - 42
 - 43
 - 44


In [53]:
# %%
# ============================================================
# DEFINIÇÃO DOS MODELOS CANDIDATOS
# ============================================================

models = {

    "LogisticRegression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ),

    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=5,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )
}


print("=" * 60)
print("MODELOS CANDIDATOS")
print("=" * 60)

for name in models:
    print(f" - {name}")

MODELOS CANDIDATOS
 - LogisticRegression
 - RandomForest
 - XGBoost


In [54]:
# %%
# ============================================================
# TREINAMENTO DOS MODELOS
# ============================================================

trained_models = {}

print("=" * 60)
print("TREINAMENTO DOS MODELOS")
print("=" * 60)

for model_name, model in models.items():

    print(f"\nTreinando: {model_name}")

    model.fit(
        X_train,
        y_train
    )

    trained_models[model_name] = model

    print(f"✓ {model_name} treinado com sucesso.")

print("\n✓ Todos os modelos foram treinados.")

TREINAMENTO DOS MODELOS

Treinando: LogisticRegression
✓ LogisticRegression treinado com sucesso.

Treinando: RandomForest
✓ RandomForest treinado com sucesso.

Treinando: XGBoost
✓ XGBoost treinado com sucesso.

✓ Todos os modelos foram treinados.


In [55]:
# %%
# ============================================================
# PREVISÕES NO CONJUNTO DE TESTE
# ============================================================

predictions = {}

for model_name, model in trained_models.items():

    y_pred = model.predict(X_test)

    y_prob = model.predict_proba(X_test)[:, 1]

    predictions[model_name] = {
        "y_pred": y_pred,
        "y_prob": y_prob
    }

    print(
        f"✓ Previsões geradas para {model_name}"
    )

✓ Previsões geradas para LogisticRegression
✓ Previsões geradas para RandomForest
✓ Previsões geradas para XGBoost


In [56]:
# %%
# ============================================================
# AVALIAÇÃO DOS MODELOS
# ============================================================

results = []

for model_name in trained_models:

    y_pred = predictions[model_name]["y_pred"]
    y_prob = predictions[model_name]["y_prob"]

    result = {
        "Modelo": model_name,

        "Accuracy": accuracy_score(
            y_test,
            y_pred
        ),

        "Precision": precision_score(
            y_test,
            y_pred,
            zero_division=0
        ),

        "Recall": recall_score(
            y_test,
            y_pred,
            zero_division=0
        ),

        "F1": f1_score(
            y_test,
            y_pred,
            zero_division=0
        ),

        "ROC_AUC": roc_auc_score(
            y_test,
            y_prob
        )
    }

    results.append(result)


comparison_df = pd.DataFrame(results)

comparison_df = comparison_df.sort_values(
    by="F1",
    ascending=False
).reset_index(drop=True)


print("=" * 60)
print("COMPARAÇÃO DOS MODELOS")
print("=" * 60)

display(
    comparison_df.round(4)
)

COMPARAÇÃO DOS MODELOS


,Modelo,Accuracy,Precision,Recall,F1,ROC_AUC
0,RandomForest,0.7687,0.5493,0.7043,0.6172,0.8322
1,LogisticRegression,0.7402,0.5061,0.7769,0.6129,0.8397
2,XGBoost,0.7879,0.6267,0.4919,0.5512,0.8371


In [57]:
# %%
# ============================================================
# CONFIGURAÇÃO DA VALIDAÇÃO CRUZADA
# ============================================================

from sklearn.model_selection import (
    StratifiedKFold,
    GridSearchCV
)

print("=" * 60)
print("VALIDAÇÃO CRUZADA")
print("=" * 60)

cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("\nEstratégia:")
print(" - StratifiedKFold")
print(" - Número de folds: 5")
print(" - Shuffle: True")
print(" - Random state: 42")

print("\n✓ Validação cruzada configurada.")

VALIDAÇÃO CRUZADA

Estratégia:
 - StratifiedKFold
 - Número de folds: 5
 - Shuffle: True
 - Random state: 42

✓ Validação cruzada configurada.


In [58]:
# %%
# ============================================================
# TUNING - REGRESSÃO LOGÍSTICA
# ============================================================

print("=" * 60)
print("TUNING - REGRESSÃO LOGÍSTICA")
print("=" * 60)

logistic_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

logistic_params = {
    "C": [
        0.01,
        0.1,
        1,
        10
    ]
}

logistic_grid = GridSearchCV(
    estimator=logistic_model,
    param_grid=logistic_params,
    scoring="f1",
    cv=cv_strategy,
    n_jobs=-1,
    refit=True
)

logistic_grid.fit(
    X_train,
    y_train
)

print("\n✓ Tuning concluído.")

print("\nMelhores parâmetros:")
print(logistic_grid.best_params_)

print("\nMelhor F1 médio na validação cruzada:")
print(f"{logistic_grid.best_score_:.4f}")

TUNING - REGRESSÃO LOGÍSTICA

✓ Tuning concluído.

Melhores parâmetros:
{'C': 0.01}

Melhor F1 médio na validação cruzada:
0.6305


In [59]:
# %%
# ============================================================
# TUNING - RANDOM FOREST
# ============================================================

print("=" * 60)
print("TUNING - RANDOM FOREST")
print("=" * 60)

rf_model = RandomForestClassifier(
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_params = {
    "n_estimators": [
        200,
        300
    ],

    "max_depth": [
        8,
        12,
        None
    ],

    "min_samples_split": [
        2,
        5
    ],

    "min_samples_leaf": [
        1,
        2
    ]
}

rf_grid = GridSearchCV(
    estimator=rf_model,
    param_grid=rf_params,
    scoring="f1",
    cv=cv_strategy,
    n_jobs=-1,
    refit=True
)

rf_grid.fit(
    X_train,
    y_train
)

print("\n✓ Tuning concluído.")

print("\nMelhores parâmetros:")
print(rf_grid.best_params_)

print("\nMelhor F1 médio na validação cruzada:")
print(f"{rf_grid.best_score_:.4f}")

TUNING - RANDOM FOREST

✓ Tuning concluído.

Melhores parâmetros:
{'max_depth': 8, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}

Melhor F1 médio na validação cruzada:
0.6388


In [60]:
# %%
# ============================================================
# TUNING - XGBOOST
# ============================================================

print("=" * 60)
print("TUNING - XGBOOST")
print("=" * 60)

xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_params = {
    "n_estimators": [
        200,
        300
    ],

    "max_depth": [
        3,
        4,
        5
    ],

    "learning_rate": [
        0.03,
        0.05,
        0.1
    ],

    "subsample": [
        0.8,
        1.0
    ]
}

xgb_grid = GridSearchCV(
    estimator=xgb_model,
    param_grid=xgb_params,
    scoring="f1",
    cv=cv_strategy,
    n_jobs=-1,
    refit=True
)

xgb_grid.fit(
    X_train,
    y_train
)

print("\n✓ Tuning concluído.")

print("\nMelhores parâmetros:")
print(xgb_grid.best_params_)

print("\nMelhor F1 médio na validação cruzada:")
print(f"{xgb_grid.best_score_:.4f}")

TUNING - XGBOOST

✓ Tuning concluído.

Melhores parâmetros:
{'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 200, 'subsample': 1.0}

Melhor F1 médio na validação cruzada:
0.5957


In [24]:
# %%
# ============================================================
# COMPARAÇÃO DO TUNING
# ============================================================

tuning_results = pd.DataFrame({
    "Modelo": [
        "LogisticRegression",
        "RandomForest",
        "XGBoost"
    ],

    "F1_CV": [
        logistic_grid.best_score_,
        rf_grid.best_score_,
        xgb_grid.best_score_
    ]
})

tuning_results = tuning_results.sort_values(
    "F1_CV",
    ascending=False
).reset_index(drop=True)

print("=" * 60)
print("RESULTADO DO TUNING")
print("=" * 60)

display(
    tuning_results.round(4)
)

RESULTADO DO TUNING


,Modelo,F1_CV
0,RandomForest,0.6388
1,LogisticRegression,0.6305
2,XGBoost,0.5957


In [61]:
# %%
print("=" * 60)
print("TUNING DE HIPERPARÂMETROS — RANDOM FOREST")
print("=" * 60)

from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

# Modelo base
rf_model = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

# Espaço de hiperparâmetros
rf_param_grid = {
    "n_estimators": [200, 300, 500, 700],
    "max_depth": [None, 5, 8, 10, 15, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_features": ["sqrt", "log2", None],
    "class_weight": [None, "balanced", "balanced_subsample"]
}

# Busca aleatória com validação cruzada
rf_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=rf_param_grid,
    n_iter=30,
    scoring="f1",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

print("\nIniciando busca pelos melhores hiperparâmetros...")
print("Critério de seleção: F1")
print("Validação cruzada: 5 folds")
print("Combinações testadas: 30")

rf_search.fit(X_train, y_train)

print("\n" + "=" * 60)
print("RESULTADO DO TUNING")
print("=" * 60)

print("\nMelhor F1 médio na validação cruzada:")
print(f"{rf_search.best_score_:.4f}")

print("\nMelhores hiperparâmetros:")

for parameter, value in rf_search.best_params_.items():
    print(f"  {parameter}: {value}")

# Modelo final encontrado pelo tuning
rf_best_model = rf_search.best_estimator_

print("\n✓ Tuning do Random Forest concluído.")

TUNING DE HIPERPARÂMETROS — RANDOM FOREST

Iniciando busca pelos melhores hiperparâmetros...
Critério de seleção: F1
Validação cruzada: 5 folds
Combinações testadas: 30
Fitting 5 folds for each of 30 candidates, totalling 150 fits

RESULTADO DO TUNING

Melhor F1 médio na validação cruzada:
0.6349

Melhores hiperparâmetros:
  n_estimators: 300
  min_samples_split: 2
  min_samples_leaf: 4
  max_features: log2
  max_depth: 8
  class_weight: balanced_subsample

✓ Tuning do Random Forest concluído.


In [26]:
# %%
print("=" * 60)
print("TUNING DE HIPERPARÂMETROS — REGRESSÃO LOGÍSTICA")
print("=" * 60)

from sklearn.model_selection import RandomizedSearchCV
from sklearn.linear_model import LogisticRegression

# Modelo base
lr_model = LogisticRegression(
    max_iter=2000,
    random_state=42
)

# Espaço de hiperparâmetros
lr_param_grid = {
    "C": [0.001, 0.01, 0.05, 0.1, 0.5, 1, 2, 5, 10, 20],
    "penalty": ["l1", "l2"],
    "solver": ["liblinear", "saga"],
    "class_weight": [None, "balanced"]
}

# Busca aleatória com validação cruzada
lr_search = RandomizedSearchCV(
    estimator=lr_model,
    param_distributions=lr_param_grid,
    n_iter=20,
    scoring="f1",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

print("\nIniciando busca pelos melhores hiperparâmetros...")
print("Critério de seleção: F1")
print("Validação cruzada: 5 folds")
print("Combinações testadas: 20")

lr_search.fit(X_train, y_train)

print("\n" + "=" * 60)
print("RESULTADO DO TUNING")
print("=" * 60)

print("\nMelhor F1 médio na validação cruzada:")
print(f"{lr_search.best_score_:.4f}")

print("\nMelhores hiperparâmetros:")

for parameter, value in lr_search.best_params_.items():
    print(f"  {parameter}: {value}")

# Modelo final encontrado pelo tuning
lr_best_model = lr_search.best_estimator_

print("\n✓ Tuning da Regressão Logística concluído.")

TUNING DE HIPERPARÂMETROS — REGRESSÃO LOGÍSTICA

Iniciando busca pelos melhores hiperparâmetros...
Critério de seleção: F1
Validação cruzada: 5 folds
Combinações testadas: 20
Fitting 5 folds for each of 20 candidates, totalling 100 fits


C:\Users\Kawan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\Kawan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(



RESULTADO DO TUNING

Melhor F1 médio na validação cruzada:
0.6323

Melhores hiperparâmetros:
  solver: saga
  penalty: l1
  class_weight: balanced
  C: 1

✓ Tuning da Regressão Logística concluído.


In [62]:
# %%
# ============================================================
# TUNING COM SMOTE - XGBOOST (Otimizando para ROC-AUC)
# ============================================================
print("=" * 60)
print("TUNING - XGBOOST COM SMOTE")
print("=" * 60)

# O segredo aqui é usar o Pipeline do imblearn para que o SMOTE
# seja aplicado APENAS nos folds de treino da validação cruzada.
xgb_pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('xgb', XGBClassifier(
        objective="binary:logistic", 
        eval_metric="logloss", 
        random_state=42, 
        n_jobs=-1
    ))
])

# Param grid otimizado
xgb_param_grid = {
    "smote__k_neighbors": [3, 5, 7],
    "xgb__n_estimators": [200, 300, 500],
    "xgb__max_depth": [3, 4, 5], # Árvores mais rasas evitam overfitting em dados de SMOTE
    "xgb__learning_rate": [0.01, 0.05, 0.1],
    "xgb__subsample": [0.7, 0.8, 0.9],
    "xgb__colsample_bytree": [0.7, 0.8, 0.9],
    "xgb__min_child_weight": [3, 5, 7],
    "xgb__gamma": [0.1, 0.5, 1, 2] # Gamma maior para forçar conservadorismo
}

# Estratégia de validação
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

xgb_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=xgb_param_grid,
    n_iter=30,
    scoring="roc_auc", # Mudamos de F1 para ROC-AUC para maximizar a separação geral
    cv=cv_strategy,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

xgb_search.fit(X_train, y_train)

print(f"\nMelhor ROC-AUC CV: {xgb_search.best_score_:.4f}")
xgb_best_model = xgb_search.best_estimator_

print("\n✓ Tuning do XGBoost com SMOTE concluído.")

TUNING - XGBOOST COM SMOTE
Fitting 5 folds for each of 30 candidates, totalling 150 fits

Melhor ROC-AUC CV: 0.8475

✓ Tuning do XGBoost com SMOTE concluído.


In [63]:
# %%
print("=" * 60)
print("AVALIAÇÃO DOS MODELOS APÓS O TUNING")
print("=" * 60)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Dicionário com os modelos tunados
tuned_models = {
    "RandomForest_Tuned": rf_best_model,
    "LogisticRegression_Tuned": lr_best_model,
    "XGBoost_Tuned": xgb_best_model
}

tuned_results = []

for model_name, model in tuned_models.items():

    print("\n" + "-" * 60)
    print(f"Avaliando: {model_name}")
    print("-" * 60)

    # Treinar modelo
    model.fit(X_train, y_train)

    # Probabilidades
    y_prob = model.predict_proba(X_test)[:, 1]

    # Threshold padrão
    y_pred = (y_prob >= 0.50).astype(int)

    # Métricas
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )
    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )
    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )
    roc_auc = roc_auc_score(
        y_test,
        y_prob
    )

    tuned_results.append({
        "Modelo": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC_AUC": roc_auc
    })

    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1       : {f1:.4f}")
    print(f"ROC-AUC  : {roc_auc:.4f}")

# Criar tabela comparativa
tuned_comparison_df = pd.DataFrame(tuned_results)

# Ordenar pelo F1
tuned_comparison_df = (
    tuned_comparison_df
    .sort_values("F1", ascending=False)
    .reset_index(drop=True)
)

print("\n" + "=" * 60)
print("COMPARATIVO DOS MODELOS TUNADOS")
print("=" * 60)

display(
    tuned_comparison_df.round(4)
)

AVALIAÇÃO DOS MODELOS APÓS O TUNING

------------------------------------------------------------
Avaliando: RandomForest_Tuned
------------------------------------------------------------
Accuracy : 0.7601
Precision: 0.5331
Recall   : 0.7581
F1       : 0.6260
ROC-AUC  : 0.8423

------------------------------------------------------------
Avaliando: LogisticRegression_Tuned
------------------------------------------------------------


C:\Users\Kawan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\Kawan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Accuracy : 0.7402
Precision: 0.5061
Recall   : 0.7769
F1       : 0.6129
ROC-AUC  : 0.8398

------------------------------------------------------------
Avaliando: XGBoost_Tuned
------------------------------------------------------------
Accuracy : 0.7858
Precision: 0.5794
Recall   : 0.6962
F1       : 0.6325
ROC-AUC  : 0.8415

COMPARATIVO DOS MODELOS TUNADOS


,Modelo,Accuracy,Precision,Recall,F1,ROC_AUC
0,XGBoost_Tuned,0.7858,0.5794,0.6962,0.6325,0.8415
1,RandomForest_Tuned,0.7601,0.5331,0.7581,0.6260,0.8423
2,LogisticRegression_Tuned,0.7402,0.5061,0.7769,0.6129,0.8398


In [64]:
# %%
print("=" * 60)
print("ANÁLISE DE THRESHOLD DOS MODELOS TUNADOS")
print("=" * 60)

thresholds = np.arange(0.30, 0.71, 0.05)

threshold_results = []

for model_name, model in tuned_models.items():

    print(f"\nAnalisando: {model_name}")

    # Probabilidades no conjunto de teste
    y_prob = model.predict_proba(X_test)[:, 1]

    for threshold in thresholds:

        y_pred = (
            y_prob >= threshold
        ).astype(int)

        threshold_results.append({
            "Modelo": model_name,
            "Threshold": round(threshold, 2),
            "Precision": precision_score(
                y_test,
                y_pred,
                zero_division=0
            ),
            "Recall": recall_score(
                y_test,
                y_pred,
                zero_division=0
            ),
            "F1": f1_score(
                y_test,
                y_pred,
                zero_division=0
            ),
            "Accuracy": accuracy_score(
                y_test,
                y_pred
            )
        })

threshold_df = pd.DataFrame(
    threshold_results
)

print("\n" + "=" * 60)
print("MELHOR THRESHOLD POR MODELO")
print("=" * 60)

best_thresholds = (
    threshold_df
    .sort_values(
        ["Modelo", "F1", "Recall"],
        ascending=[True, False, False]
    )
    .groupby("Modelo")
    .head(1)
    .reset_index(drop=True)
)

display(
    best_thresholds.round(4)
)

print("\n" + "=" * 60)
print("TOP RESULTADOS POR F1")
print("=" * 60)

top_thresholds = (
    threshold_df
    .sort_values(
        ["F1", "Recall"],
        ascending=[False, False]
    )
    .head(15)
)

display(
    top_thresholds.round(4)
)

ANÁLISE DE THRESHOLD DOS MODELOS TUNADOS

Analisando: RandomForest_Tuned

Analisando: LogisticRegression_Tuned

Analisando: XGBoost_Tuned

MELHOR THRESHOLD POR MODELO


,Modelo,Threshold,Precision,Recall,F1,Accuracy
0,LogisticRegression_Tuned,0.55,0.5342,0.7339,0.6183,0.7601
1,RandomForest_Tuned,0.45,0.5126,0.8172,0.6301,0.7459
2,XGBoost_Tuned,0.45,0.5629,0.7339,0.6371,0.7786



TOP RESULTADOS POR F1


,Modelo,Threshold,Precision,Recall,F1,Accuracy
21,XGBoost_Tuned,0.45,0.5629,0.7339,0.6371,0.7786
22,XGBoost_Tuned,0.50,0.5794,0.6962,0.6325,0.7858
3,RandomForest_Tuned,0.45,0.5126,0.8172,0.6301,0.7459
4,RandomForest_Tuned,0.50,0.5331,0.7581,0.6260,0.7601
20,XGBoost_Tuned,0.40,0.5288,0.7661,0.6257,0.7573
5,RandomForest_Tuned,0.55,0.5622,0.7043,0.6253,0.7765
6,RandomForest_Tuned,0.60,0.5941,0.6532,0.6223,0.7900
14,LogisticRegression_Tuned,0.55,0.5342,0.7339,0.6183,0.7601
23,XGBoost_Tuned,0.55,0.5916,0.6425,0.6160,0.7879
13,LogisticRegression_Tuned,0.50,0.5061,0.7769,0.6129,0.7402


In [ ]:
# %%
# ============================================================
# OTIMIZAÇÃO DO THRESHOLD (Buscando o "Sweet Spot")
# ============================================================
from sklearn.model_selection import cross_val_predict

print("=" * 60)
print("BUSCA DO THRESHOLD IDEAL (OOF)")
print("=" * 60)

# Gerando probabilidades Out-Of-Fold com o melhor modelo
y_prob_oof = cross_val_predict(
    xgb_best_model, 
    X_train, 
    y_train, 
    cv=cv_strategy, 
    method="predict_proba", 
    n_jobs=-1
)[:, 1]
 
thresholds = np.arange(0.35, 0.65, 0.01)
best_threshold = 0.50
best_f1_acc_balance = 0

for t in thresholds:
    y_pred = (y_prob_oof >= t).astype(int)
    acc = accuracy_score(y_train, y_pred)
    f1 = f1_score(y_train, y_pred)
    
    # Queremos F1 >= 0.68 E Acc >= 0.80. 
    # Vamos criar uma pontuação customizada que pune severamente se não atingir as metas
    score = (acc + f1)
    if acc < 0.80 or f1 < 0.68:
        score -= 1.0 # Penalidade pesada para forçar o equilíbrio
        
    if score > best_f1_acc_balance:
        best_f1_acc_balance = score
        best_threshold = t

print(f"\nThreshold Selecionado: {best_threshold:.2f}")
print("✓ Threshold otimizado para equilibrar Acurácia e F1.")

BUSCA DO THRESHOLD IDEAL (OOF)

Threshold Selecionado: 0.51
✓ Threshold otimizado para equilibrar Acurácia e F1.


In [66]:
# %%
# ============================================================
# AVALIAÇÃO FINAL NO TESTE
# ============================================================
print("=" * 60)
print("AVALIAÇÃO FINAL (CONJUNTO DE TESTE INTOCADO)")
print("=" * 60)

# Treinar o modelo final com todo o dado de treino
xgb_best_model.fit(X_train, y_train)

# Prever no teste usando o threshold otimizado
y_test_prob = xgb_best_model.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_prob >= best_threshold).astype(int)

test_accuracy = accuracy_score(y_test, y_test_pred)
test_f1 = f1_score(y_test, y_test_pred)
test_roc_auc = roc_auc_score(y_test, y_test_prob)

print(f"\nAUC-ROC  : {test_roc_auc:.4f} (Meta: >= 0.82)")
print(f"Accuracy : {test_accuracy:.4f} (Meta: >= 0.80)")
print(f"F1-Score : {test_f1:.4f} (Meta: >= 0.68)")

print("\n" + "-" * 60)
if test_roc_auc >= 0.82 and test_accuracy >= 0.80 and test_f1 >= 0.68:
    print("🎉 PARABÉNS! Todas as métricas foram atingidas!")
else:
    print("⚠ Ainda não atingimos todas as métricas.")
    print("Próximo passo sugerido: Retornar à Etapa 3 (Engenharia de Atributos).")

AVALIAÇÃO FINAL (CONJUNTO DE TESTE INTOCADO)

AUC-ROC  : 0.8415 (Meta: >= 0.82)
Accuracy : 0.7858 (Meta: >= 0.80)
F1-Score : 0.6298 (Meta: >= 0.68)

------------------------------------------------------------
⚠ Ainda não atingimos todas as métricas.
Próximo passo sugerido: Retornar à Etapa 3 (Engenharia de Atributos).
